In [1]:
import pandas as pd
from pycaret.classification import *

In [2]:
file_path = 'D:\\SEGP-G6\\notebooks\\Churn_Modelling.csv'
data = pd.read_csv(file_path)

In [3]:
# Remove duplicate rows
data = data.drop_duplicates()
# Remove Leading/Trailing Spaces
data.columns = data.columns.str.strip()

In [ ]:
# Define maximum allowed categories (less than 1% of rows)
max_categories_threshold = int(len(data) * 0.01)

# Identify categorical columns (object or category dtypes) that meet the threshold
eligible_cat_features = [
    col for col in data.select_dtypes(include=['object', 'category', 'bool']).columns
    if data[col].nunique() <= max_categories_threshold]

print("Eligible categorical features for encoding:", eligible_cat_features)

# Encode each eligible categorical column using pd.factorize
for col in eligible_cat_features:
    data[col], _ = pd.factorize(data[col])


Eligible categorical features for encoding: ['Geography', 'Gender']


In [5]:
target = 'null'
# Transit target from front end

# Detect target column if not provided
if target == 'null' or target not in data.columns:
    def detect_target_column(df):
        possible_target = None
        for col in df.columns:
            unique_values = df[col].nunique()
            # Check if the column has no missing values and is suitable for classification
            if unique_values == 2 and df[col].isnull().sum() == 0:  # Binary classification without missing values
                possible_target = col
            elif 2 < unique_values < len(data) * 0.01 and df[col].isnull().sum() == 0:  # Multi-class classification without missing values
                possible_target = col
        return possible_target # Target column is more possible on the right side
    
    target = detect_target_column(data)

# find if target is found
if target:
    print(f"Detected target column: {target}")
else:
    print("No suitable target column detected. Please specify it manually.")

if target in eligible_cat_features:
    eligible_cat_features.remove(target)

Detected target column: Exited


In [6]:
import re

def detect_ignore_features(df, keywords=None):
    if keywords is None:
        # Only ignore columns that are very likely identifiers or irrelevant
        keywords = ['row', 'id', 'name']
        
    ignore_features = []
    
    for col in df.columns:
        # Check if column name contains any of the keywords
        if any(re.search(keyword, col, flags=re.IGNORECASE) for keyword in keywords):
            ignore_features.append(col)
    
    return ignore_features

# Usage with your DataFrame
ignore_features = detect_ignore_features(data)
print("Automatically detected ignore features:", ignore_features)


Automatically detected ignore features: ['RowNumber', 'CustomerId', 'Surname']


In [ ]:
numeric_imputation = user_input

In [ ]:
clf = setup(
    data=data,
    target=target,
    
    # Data cleaning configuration
    numeric_imputation="mean",  # Fill missing numerical values with the median
    categorical_imputation='mode',  # Fill missing categorical values with the mode
    remove_multicollinearity=True,  # Remove multicollinearity features
    multicollinearity_threshold=1.0, 
    remove_outliers=True,  # Automatically handle outliers using the IQR method
    outliers_threshold=0.05,  # Outlier detection threshold
    fold_strategy='stratifiedkfold',  # Use stratified K-fold cross-validation
    categorical_features=eligible_cat_features,
    normalize=True,

    # Categorical feature handling
    ignore_features=ignore_features,  # Ignore irrelevant features
    
    # Other automation settings
    fix_imbalance=True,  # Automatically handle class imbalance
    session_id=42  # Random seed
)


,Description,Value
0,Session id,42
1,Target,Exited
2,Target type,Binary
3,Original data shape,"(10000, 14)"
4,Transformed data shape,"(13728, 13)"
5,Transformed train set shape,"(10728, 13)"
6,Transformed test set shape,"(3000, 13)"
7,Ignore features,3
8,Numeric features,8
9,Categorical features,2


In [8]:
# Get the cleaned data
cleaned_data = get_config('X_train')  # Retrieve the cleaned data (excluding the target column)
cleaned_data[target] = get_config('y_train')  # Re-add the target column

# Save the cleaned data
output_file_path = 'D:\\SEGP-G6\\notebooks\\output_PyCaret.csv'
cleaned_data.to_csv(output_file_path, index=False)

print(f"\nData cleaning completed. Saved to: {output_file_path}")



Data cleaning completed. Saved to: D:\SEGP-G6\notebooks\output_PyCaret.csv


In [9]:
# Retrieve and compare all models
compare_models(sort='AUC')

# Get the comparison results
model_comparison = pull()
print(model_comparison)


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
gbc,Gradient Boosting Classifier,0.8553,0.8545,0.5161,0.6950,0.5918,0.5063,0.5149,0.4770
catboost,CatBoost Classifier,0.8579,0.8544,0.5070,0.7128,0.5917,0.5087,0.5200,2.8160
lightgbm,Light Gradient Boosting Machine,0.8519,0.8492,0.4880,0.6953,0.5728,0.4866,0.4981,0.4150
rf,Random Forest Classifier,0.8523,0.8418,0.4873,0.6973,0.5731,0.4873,0.4990,0.2780
et,Extra Trees Classifier,0.8513,0.8375,0.4719,0.7006,0.5634,0.4780,0.4918,0.2200
ada,Ada Boost Classifier,0.8429,0.8359,0.5364,0.6369,0.5812,0.4855,0.4889,0.2010
xgboost,Extreme Gradient Boosting,0.8440,0.8318,0.4810,0.6621,0.5564,0.4647,0.4739,0.2840
qda,Quadratic Discriminant Analysis,0.6553,0.7714,0.7561,0.3487,0.4730,0.2699,0.3172,0.0780
knn,K Neighbors Classifier,0.7670,0.7606,0.5736,0.4442,0.5002,0.3516,0.3568,0.9840
lr,Logistic Regression,0.7020,0.7547,0.6886,0.3745,0.4850,0.3003,0.3280,1.2590


                                    Model  Accuracy     AUC  Recall   Prec.  \
gbc          Gradient Boosting Classifier    0.8553  0.8545  0.5161  0.6950   
catboost              CatBoost Classifier    0.8579  0.8544  0.5070  0.7128   
lightgbm  Light Gradient Boosting Machine    0.8519  0.8492  0.4880  0.6953   
rf               Random Forest Classifier    0.8523  0.8418  0.4873  0.6973   
et                 Extra Trees Classifier    0.8513  0.8375  0.4719  0.7006   
ada                  Ada Boost Classifier    0.8429  0.8359  0.5364  0.6369   
xgboost         Extreme Gradient Boosting    0.8440  0.8318  0.4810  0.6621   
qda       Quadratic Discriminant Analysis    0.6553  0.7714  0.7561  0.3487   
knn                K Neighbors Classifier    0.7670  0.7606  0.5736  0.4442   
lr                    Logistic Regression    0.7020  0.7547  0.6886  0.3745   
nb                            Naive Bayes    0.7249  0.7547  0.6079  0.3881   
ridge                    Ridge Classifier    0.7031 